In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel("../data/cleaned_eda.xlsx")

In [3]:
df.head()

,Delivery_person_Age,Delivery_person_Ratings,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,Time_taken(min),...,order_dayofweek,order_day_name,is_weekend,order_hour,order_minute,time_category,is_peak_hour,is_late_night,is_rush_hour,season
0,37,4.9,Sunny,High,2,Snack,motorcycle,0,No,24,...,5,Saturday,1,11,30,Morning,0,0,0,Spring
1,34,4.5,Stormy,Jam,2,Snack,scooter,1,No,33,...,4,Friday,0,19,45,Evening,1,0,0,Spring
2,23,4.4,Sandstorms,Low,0,Drinks,motorcycle,1,No,26,...,5,Saturday,1,8,30,Morning,0,0,1,Spring
3,38,4.7,Sunny,Medium,0,Buffet,motorcycle,1,No,21,...,1,Tuesday,0,18,0,Evening,0,0,1,Spring
4,32,4.6,Cloudy,High,1,Snack,scooter,1,No,30,...,5,Saturday,1,13,30,Afternoon,1,0,0,Spring


In [5]:
"""
CORRECT APPROACH: Feature Selection That Catches Non-Linear Relationships
This combines statistical tests, model-based importance, and domain-aware selection
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression, f_regression
from sklearn.preprocessing import LabelEncoder, KBinsDiscretizer
import warnings
warnings.filterwarnings('ignore')

class RobustFeatureSelector:
    """
    The CORRECT way to find important features, including those with 
    non-linear relationships that correlation misses.
    """
    
    def __init__(self, target_col, random_state=42):
        self.target_col = target_col
        self.random_state = random_state
        self.results = None
        
    def prepare_features(self, df, categorical_cols=None, exclude_cols=None):
        """Prepare features with proper encoding."""
        X = df.drop(columns=[self.target_col])
        y = df[self.target_col]
        
        if exclude_cols:
            X = X.drop(columns=exclude_cols)
        
        # Identify categorical columns
        if categorical_cols is None:
            categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
        
        # Encode categorical variables
        self.encoders = {}
        for col in categorical_cols:
            le = LabelEncoder()
            # Handle NaN if present
            X[col] = X[col].fillna('MISSING').astype(str)
            X[col] = le.fit_transform(X[col])
            self.encoders[col] = le
        
        self.X = X
        self.y = y
        self.feature_names = X.columns.tolist()
        print(f"✅ Prepared {len(self.feature_names)} features")
        return self
    
    # ============ METHOD 1: Mutual Information (Captures Non-Linear) ============
    def mutual_information_selection(self, k=20):
        """
        Mutual Information captures ANY relationship (linear OR non-linear).
        This is CRITICAL for finding features correlation misses.
        """
        mi_scores = mutual_info_regression(self.X, self.y, random_state=self.random_state)
        
        mi_df = pd.DataFrame({
            'feature': self.feature_names,
            'mutual_info': mi_scores
        }).sort_values('mutual_info', ascending=False)
        
        print(f"\n📊 Mutual Information Scores (Top {k}):")
        print(mi_df.head(k).to_string(index=False))
        
        return mi_df
    
    # ============ METHOD 2: F-regression (Linear Only - For Comparison) ============
    def f_regression_selection(self, k=20):
        """
        F-regression tests LINEAR relationship only.
        Features with LOW f-score but HIGH mutual info are your HIDDEN GEMS!
        """
        f_scores, p_values = f_regression(self.X, self.y)
        
        f_df = pd.DataFrame({
            'feature': self.feature_names,
            'f_score': f_scores,
            'p_value': p_values
        }).sort_values('f_score', ascending=False)
        
        print(f"\n📊 F-Regression Scores (Top {k}):")
        print(f_df.head(k).to_string(index=False))
        
        return f_df
    
    # ============ METHOD 3: Boruta-like Importance (Statistical Significance) ============
    def rf_importance_with_shadows(self, n_estimators=100):
        """
        Creates shadow features to determine which features are TRULY important
        (statistically significant, not just by chance).
        """
        # Create shadow features (random permutations)
        np.random.seed(self.random_state)
        X_shadow = self.X.copy()
        
        for col in X_shadow.columns:
            X_shadow[f'shadow_{col}'] = np.random.permutation(X_shadow[col].values)
        
        # Train Random Forest
        rf = RandomForestRegressor(n_estimators=n_estimators, random_state=self.random_state, n_jobs=-1)
        rf.fit(X_shadow, self.y)
        
        # Get importances
        importances = pd.DataFrame({
            'feature': X_shadow.columns,
            'importance': rf.feature_importances_
        })
        
        # Separate real vs shadow
        real_imp = importances[~importances['feature'].str.startswith('shadow_')].copy()
        shadow_imp = importances[importances['feature'].str.startswith('shadow_')].copy()
        
        # Find max shadow importance (threshold for significance)
        max_shadow = shadow_imp['importance'].max()
        
        real_imp['significant'] = real_imp['importance'] > max_shadow
        real_imp = real_imp.sort_values('importance', ascending=False)
        
        print(f"\n📊 Random Forest Importance with Shadow Test:")
        print(f"   Threshold (max shadow importance): {max_shadow:.4f}")
        print(f"   Significant features: {real_imp['significant'].sum()} / {len(real_imp)}")
        print("\n   Top Significant Features:")
        print(real_imp[real_imp['significant']].head(15).to_string(index=False))
        
        return real_imp
    
    # ============ METHOD 4: Non-Linear Pattern Detection ============
    def detect_non_linear_features(self, n_bins=10, threshold=0.15):
        """
        Identify features with potential non-linear relationships.
        """
        non_linear_features = []
        
        for col in self.feature_names:
            # Discretize feature into bins
            if len(self.X[col].unique()) > n_bins:
                discretizer = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='uniform')
                X_binned = discretizer.fit_transform(self.X[[col]]).ravel()
            else:
                X_binned = self.X[col].values
            
            # Calculate variance of means across bins
            bin_means = []
            for b in sorted(np.unique(X_binned)):
                mask = X_binned == b
                if mask.sum() > 0:
                    bin_means.append(self.y[mask].mean())
            
            if len(bin_means) > 2:
                # Check for non-linearity by comparing linear fit to actual
                x_pos = np.arange(len(bin_means))
                z = np.polyfit(x_pos, bin_means, 1)
                p = np.poly1d(z)
                linear_fit = p(x_pos)
                
                # Calculate non-linearity score
                residuals = np.abs(bin_means - linear_fit)
                non_linear_score = residuals.mean() / (np.std(bin_means) + 1e-6)
                
                if non_linear_score > threshold:
                    non_linear_features.append({
                        'feature': col,
                        'non_linear_score': non_linear_score,
                        'pattern': self._describe_pattern(bin_means)
                    })
        
        print(f"\n📊 Non-Linear Features Detected ({len(non_linear_features)}):")
        for f in non_linear_features[:10]:
            print(f"   • {f['feature']}: {f['pattern']} (score={f['non_linear_score']:.2f})")
        
        return pd.DataFrame(non_linear_features)
    
    def _describe_pattern(self, bin_means):
        """Describe the pattern of binned means."""
        if len(bin_means) < 3:
            return "Insufficient data"
        
        first = bin_means[0]
        mid = bin_means[len(bin_means)//2]
        last = bin_means[-1]
        
        if first < mid > last:
            return "Inverted-U (∩) - peaks in middle"
        elif first > mid < last:
            return "U-shaped (∪) - drops in middle"
        elif first < last:
            return "Increasing trend"
        elif first > last:
            return "Decreasing trend"
        else:
            return "Complex pattern"
    
    # ============ METHOD 5: Cross-Validation Feature Elimination ============
    def recursive_feature_elimination_cv(self, n_features_to_select=None, cv=5):
        """
        Use cross-validation to find optimal feature subset.
        This is the MOST RELIABLE method for feature selection.
        """
        from sklearn.feature_selection import RFECV
        
        rf = RandomForestRegressor(n_estimators=50, random_state=self.random_state, n_jobs=-1)
        
        rfecv = RFECV(
            estimator=rf,
            step=1,
            cv=cv,
            scoring='r2',
            min_features_to_select=1,
            n_jobs=-1
        )
        
        rfecv.fit(self.X, self.y)
        
        selected_features = self.X.columns[rfecv.support_].tolist()
        
        print(f"\n📊 Recursive Feature Elimination with CV:")
        print(f"   Optimal number of features: {rfecv.n_features_}")
        print(f"   Selected features: {selected_features}")
        
        # Create ranking DataFrame
        ranking_df = pd.DataFrame({
            'feature': self.feature_names,
            'ranking': rfecv.ranking_,
            'selected': rfecv.support_
        }).sort_values('ranking')
        
        return ranking_df, selected_features
    
    # ============ COMPREHENSIVE ANALYSIS ============
    def comprehensive_selection(self, top_k=20):
        """
        Run ALL methods and combine results to find the BEST features,
        especially those missed by simple correlation.
        """
        print("\n" + "="*70)
        print("COMPREHENSIVE FEATURE SELECTION ANALYSIS")
        print("="*70)
        
        # Run all methods
        mi_df = self.mutual_information_selection(top_k)
        f_df = self.f_regression_selection(top_k)
        rf_df = self.rf_importance_with_shadows()
        nl_df = self.detect_non_linear_features()
        rfecv_df, selected = self.recursive_feature_elimination_cv()
        
        # Combine results
        combined = pd.DataFrame({'feature': self.feature_names})
        
        # Add mutual info
        combined = combined.merge(mi_df, on='feature', how='left')
        
        # Add f-score
        combined = combined.merge(f_df[['feature', 'f_score']], on='feature', how='left')
        
        # Add RF importance
        combined = combined.merge(rf_df[['feature', 'importance']], on='feature', how='left')
        combined.rename(columns={'importance': 'rf_importance'}, inplace=True)
        
        # Add significance flag
        if 'significant' in rf_df.columns:
            combined = combined.merge(rf_df[['feature', 'significant']], on='feature', how='left')
        
        # Add non-linear flag
        combined['non_linear'] = combined['feature'].isin(nl_df['feature']) if len(nl_df) > 0 else False
        
        # Add RFECV ranking
        combined = combined.merge(rfecv_df[['feature', 'ranking', 'selected']], on='feature', how='left')
        
        # Calculate DISCREPANCY (features correlation misses)
        # Normalize scores for fair comparison
        combined['mutual_info_norm'] = (combined['mutual_info'] - combined['mutual_info'].min()) / (combined['mutual_info'].max() - combined['mutual_info'].min())
        combined['f_score_norm'] = (combined['f_score'] - combined['f_score'].min()) / (combined['f_score'].max() - combined['f_score'].min())
        
        # Discrepancy = MI > F-score (captures non-linear relationships)
        combined['discrepancy'] = combined['mutual_info_norm'] - combined['f_score_norm']
        
        # Sort by RF importance (or mutual info if RF not available)
        sort_col = 'rf_importance' if 'rf_importance' in combined.columns else 'mutual_info'
        combined = combined.sort_values(sort_col, ascending=False)
        
        # Final recommendation
        print("\n" + "="*70)
        print("🎯 FINAL FEATURE RECOMMENDATIONS")
        print("="*70)
        
        # Tier 1: High importance AND statistically significant
        tier1 = combined[combined.get('significant', True) == True].head(top_k // 2)
        
        # Tier 2: High discrepancy (correlation missed these!)
        tier2 = combined.nlargest(top_k // 2, 'discrepancy')
        tier2 = tier2[~tier2['feature'].isin(tier1['feature'])]
        
        # Tier 3: Selected by RFECV
        tier3 = combined[combined['selected'] == True].head(top_k)
        tier3 = tier3[~tier3['feature'].isin(tier1['feature']) & ~tier3['feature'].isin(tier2['feature'])]
        
        print("\n🏆 TIER 1 - Essential Features (High importance & statistically significant):")
        for feat in tier1['feature'].values[:10]:
            mi = tier1[tier1['feature'] == feat]['mutual_info'].values[0] if 'mutual_info' in tier1.columns else 0
            print(f"   ✓ {feat} (MI={mi:.3f})")
        
        print("\n💎 TIER 2 - Hidden Gems (Low linear correlation but high predictive power):")
        for feat in tier2['feature'].values[:10]:
            mi = tier2[tier2['feature'] == feat]['mutual_info'].values[0] if 'mutual_info' in tier2.columns else 0
            f_score = tier2[tier2['feature'] == feat]['f_score'].values[0] if 'f_score' in tier2.columns else 0
            print(f"   ★ {feat} (MI={mi:.3f}, F-score={f_score:.1f}) - Correlation MISSED this!")
        
        print("\n📌 TIER 3 - Additional Good Features (Selected by cross-validation):")
        for feat in tier3['feature'].values[:5]:
            print(f"   • {feat}")
        
        return combined, tier1, tier2, tier3


# ============================================================================
# QUICK START - Copy this into your project
# ============================================================================

def find_best_features(df, target_col, categorical_cols=None, exclude_cols=None):
    """
    ONE FUNCTION to find the BEST features for your model.
    Automatically identifies features that correlation would miss.
    
    Returns: DataFrame with feature rankings and recommendations
    """
    selector = RobustFeatureSelector(target_col)
    selector.prepare_features(df, categorical_cols, exclude_cols)
    results, tier1, tier2, tier3 = selector.comprehensive_selection()
    
    return {
        'all_features': results,
        'essential_features': tier1['feature'].tolist(),
        'hidden_gems': tier2['feature'].tolist(),
        'recommended_features': tier1['feature'].tolist() + tier2['feature'].tolist() + tier3['feature'].tolist()
    }


# ============================================================================
# EXAMPLE USAGE
# ============================================================================

if __name__ == "__main__":
    # For your delivery data:
    results_dict = find_best_features(df, 'Time_taken(min)')
    essential = results_dict['essential_features']
    hidden_gems = results_dict['hidden_gems']
    print(f"\n✅ Essential features: {essential}")
    print(f"💎 Hidden gems (correlation missed): {hidden_gems}")
    
    print("\n" + "="*70)
    print("CORRECT APPROACH SUMMARY")
    print("="*70)
    print("""
    The CORRECT approach to find important features (especially low-correlation ones):
    
    1. Mutual Information - Captures ANY relationship (linear OR non-linear)
    2. F-regression - Captures ONLY linear (for comparison)
    3. Boruta-like Shadow Features - Tests statistical significance
    4. Non-linear Pattern Detection - Identifies U-shaped, inverted-U patterns
    5. RFECV - Cross-validated feature elimination
    
    KEY INSIGHT: Features with HIGH Mutual Info but LOW F-score are your
    "hidden gems" - they have non-linear relationships with the target!
    
    USE THIS instead of simple correlation for feature selection.
    """)

✅ Prepared 24 features

COMPREHENSIVE FEATURE SELECTION ANALYSIS

📊 Mutual Information Scores (Top 20):
                 feature  mutual_info
 Delivery_person_Ratings     0.168982
              order_hour     0.134054
    Road_traffic_density     0.125935
     multiple_deliveries     0.118836
               order_day     0.080303
     Delivery_person_Age     0.078733
           time_category     0.063225
            is_peak_hour     0.061577
       Vehicle_condition     0.061549
       Weatherconditions     0.055921
                Festival     0.047895
           is_late_night     0.031306
         Type_of_vehicle     0.019459
            is_rush_hour     0.012817
         order_dayofweek     0.008138
          order_day_name     0.004180
                  season     0.002048
             order_month     0.002030
preparation_time_minutes     0.000331
              is_weekend     0.000000

📊 F-Regression Scores (Top 20):
                 feature     f_score       p_value
     multiple_

### Final cols

In [10]:
# ALL 24 features from your analysis
all_features = [
    'Delivery_person_Ratings',
    'multiple_deliveries',
    'order_hour',
    'Road_traffic_density',
    'order_day',
    'Delivery_person_Age',
    'time_category',
    'is_peak_hour',
    'Vehicle_condition',
    'Weatherconditions',
    'Festival',
    'is_late_night',
    'Type_of_vehicle',
    'is_rush_hour',
    'order_dayofweek',
    'order_day_name',
    'season',
    'order_month',
    'preparation_time_minutes',
    'delivery_city',
    'order_minute',
    'Type_of_order',
    'order_year',  # From your df.columns
    'is_weekend'   # From your df.columns (MI was 0 but keep for now)
]

# Target column
target = 'Time_taken(min)'

df_final = df[all_features + [target]].copy()

In [11]:
df_final.to_excel("../data/delivery_data_24features.xlsx", index=False)